# Importing Libraries

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

import warnings
warnings.filterwarnings('ignore')

# Connecting to Database

In [3]:
conn = sqlite3.connect('/kaggle/input/airline-data-analysis/travel.sqlite')
cursor = conn.cursor()

# List of Tables

In [4]:
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", conn)
tables

,type,name,tbl_name,rootpage,sql
0,table,aircrafts_data,aircrafts_data,2,CREATE TABLE aircrafts_data (\r\n aircraft_...
1,table,airports_data,airports_data,3,CREATE TABLE airports_data (\r\n airport_co...
2,table,boarding_passes,boarding_passes,4,CREATE TABLE boarding_passes (\r\n ticket_n...
3,table,bookings,bookings,5,CREATE TABLE bookings (\r\n book_ref charac...
4,table,flights,flights,6,CREATE TABLE flights (\r\n flight_id intege...
5,table,seats,seats,7,CREATE TABLE seats (\r\n aircraft_code char...
6,table,ticket_flights,ticket_flights,8,CREATE TABLE ticket_flights (\r\n ticket_no...
7,table,tickets,tickets,9,CREATE TABLE tickets (\r\n ticket_no charac...


In [5]:
tables.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      8 non-null      object
 1   name      8 non-null      object
 2   tbl_name  8 non-null      object
 3   rootpage  8 non-null      int64 
 4   sql       8 non-null      object
dtypes: int64(1), object(4)
memory usage: 448.0+ bytes


# Data Exploration

In [ ]:
tables.describe()

In [ ]:
aircrafts_data = pd.read_sql_query("select * from aircrafts_data", conn)
aircrafts_data

In [ ]:
aircrafts_data.to_csv("aircrafts_data")

### Observation
- Seems like we have some data in multiple languages.
- We need to keep only understandable data

In [ ]:
aircrafts_data['model'] = aircrafts_data['model'].apply(lambda x: json.loads(x)['en'])
aircrafts_data

In [ ]:
# aircrafts_data['modela'] = aircrafts_data['model'].apply(lambda x: json.loads(x)['en'])
# aircrafts_data[:5]

In [ ]:
airports_data = pd.read_sql_query("select * from airports_data", conn)
airports_data

In [ ]:
airports_data['airport_name'] = airports_data['airport_name'].apply(lambda x: json.loads(x)['en'])
airports_data['city'] = airports_data['city'].apply(lambda x: json.loads(x)['en'])

In [ ]:
airports_data.to_csv('airports_data')

In [ ]:
boarding_passes = pd.read_sql_query("select * from boarding_passes", conn)
boarding_passes.to_csv('boarding_passes')

In [ ]:
bookings = pd.read_sql_query("select * from bookings", conn)
bookings.to_csv('bookings')

In [ ]:
flights = pd.read_sql_query("select * from flights", conn)
flights.to_csv('flights')

In [ ]:
seats = pd.read_sql_query("select * from seats", conn)
seats.to_csv('seats')

In [ ]:
ticket_flights = pd.read_sql_query("select * from ticket_flights", conn)
ticket_flights.to_csv('ticket_flights')

In [ ]:
tickets = pd.read_sql_query("select * from tickets", conn)
tickets.to_csv('tickets')

In [ ]:
aircrafts_data.head()

In [ ]:
sns.set_style('darkgrid')
fig,axes = plt.subplots(figsize=(12,8))
ax = sns.barplot(x="model", y="range",data=aircrafts_data, palette ='Paired')
for container in ax.containers:
    ax.bar_label(container)
plt.title('Airplane models with their ranges')
plt.xticks(rotation=45)
plt.show()



In [ ]:
sns.set_style('whitegrid')
fig,axes = plt.subplots(figsize=(12,8))
ax = sns.barplot(x='model',y='range', data=aircrafts_data, palette = 'Paired')
for container in ax.containers:
    ax.bar_label(container)
plt.title('AirPlane Models with their ranges')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.set_style('whitegrid')
fig,axes = plt.subplots(figsize=(12,8))
ax = sns.barplot(x='model', y='range', data=aircrafts_data, palette='Paired')
for container in ax.containers:
    ax.bar_label(container)
plt.title("Airplane Models with their ranges")
plt.xticks(rotation=45)

plt.show()

### Planes having more than 100 seats

In [ ]:
df = pd.read_sql_query("""select aircraft_code, count(*) as num_seats from seats
                        group by aircraft_code having num_seats >100""", conn)
df.head()

df.to_csv('aircraft_seats.csv')

In [ ]:
sns.set_style('whitegrid')
fig,axes = plt.subplots(figsize=(12,8))
ax = sns.barplot(x='aircraft_code',y='num_seats', data=df, palette = 'flare')
for container in ax.containers:
    ax.bar_label(container)
plt.title('AirCraft codes Vs Number of Seats')
plt.xticks(rotation=45)
plt.show()

In [ ]:
crafts = pd.read_sql("""SELECT aircraft_code, model->'en'
                        FROM aircrafts_data
                        where aircraft_code IN (319, 320, 321, 733, 763, 773);""", conn)
crafts

In [ ]:
crafts = pd.read_sql("""SELECT aircraft_code, model->'en'
                        FROM aircrafts_data
                        where aircraft_code IN (319, 320, 321, 733, 763, 773);""", conn)
crafts

### Observation
- Here we successfully derived the names of airplanes using their codes
- So it seems like " Boeing 777-300 " is having maximum number of seats (402).

### Number of tickets booked and total amount earned changed with the time

In [ ]:
tickets = pd.read_sql_query("""select * from tickets inner join bookings
                    on tickets.book_ref = bookings.book_ref""", conn)

tickets['book_date'] = pd.to_datetime(tickets['book_date'])
tickets['date'] = tickets['book_date'].dt.date
tickets_count = tickets.groupby('date')[['date']].count()
plt.figure(figsize=(18,6))
plt.plot(tickets_count.index, tickets_count['date'], color='green', scalex=True, marker = "*")
plt.title('Number of Tickets Booked on Each Date', fontsize=30)
plt.xlabel('Date', fontsize=20)
plt.ylabel('Number of Tickets', fontsize=20)
plt.grid('b')
plt.show()

### Observation
- Utilized a line chart visualization to analyze the trend of ticket bookings and revenue earned.
- The number of tickets booked showed a gradual increase from June 22nd to July 7th.
- From July 8th until August, ticket bookings remained relatively stable with a noticeable peak in bookings on a single day.
- The revenue earned by the company is closely correlated with the number of tickets booked.
- The total revenue earned followed a similar trend throughout the analyzed time period.
- Further exploration of the factors contributing to the peak in ticket bookings could help increase overall revenue and optimize operational strategies.

In [ ]:
bookings = pd.read_sql_query("select * from bookings", conn)

bookings['book_date'] = pd.to_datetime(bookings['book_date'])
bookings['date'] = bookings['book_date'].dt.date
booking_amount = bookings.groupby('date')[['total_amount']].sum()

plt.figure(figsize=(18,6))
plt.plot(booking_amount.index, booking_amount['total_amount'],color='orange',scalex=True, marker = '*')
plt.title('Number of Tickets Booked on Each Date', fontsize=30)
plt.xlabel('Date', fontsize=20)
plt.ylabel('Total Amount Earned', fontsize=20)
plt.grid('b')
plt.show()

## Fare Distribution for the Flights

In [ ]:
df = pd.read_sql_query("""
                        SELECT  fare_conditions, aircraft_code, avg(amount)
                        FROM ticket_flights JOIN flights
                        ON ticket_flights.flight_id = flights.flight_id
                        GROUP BY aircraft_code, fare_conditions""", conn)
df.head()

In [ ]:
# Saving DF to a csv file 
df.to_csv('fare_avg_amount.csv')

In [ ]:
#sns.set_style('whitegrid')
fig, axes = plt.subplots(figsize=(10,8))
ax = sns.barplot(x ='aircraft_code', y='avg(amount)', hue ='fare_conditions',data=df, palette='flare')
for container in ax.containers:
    ax.bar_label(container)
plt.title("Average Seat's price per class")
plt.xticks(rotation=45)
plt.xlabel("Aircraft Code", fontsize=15)
plt.ylabel("Average Amount", fontsize=15)
plt.show()

    


In [ ]:
crafts = pd.read_sql("""SELECT aircraft_code, model->'en'
                        FROM aircrafts_data
                        WHERE aircraft_code IN (319, 321, 733, 763, 773, 'CN1', 'CR2', 'SU9');""", conn)
crafts

### Observation
- Here we successfully derived the names of airplanes using their codes
- So it seems like " Airbus A319-100 " is having maximum  average number of Business class seats.
- Also " Airbus A319-100 " is having maximum average number of Economy seats.
- And " Boeing 777-300 " is having maximum number of Comfort seats.

# Examining Occupancy Rate

To maximize profitability, airlines must analyze revenue streams, including overall income, average revenue per ticket, and occupancy rates. This information helps identify profitable aircraft types, itineraries, and pricing optimization opportunities. The highest total revenue is generated by the SU9 aircraft, likely due to its lower ticket prices. The CN1 aircraft has the lowest total revenue, possibly due to its limited economy class offering. Monitoring average occupancy rates helps airlines fill seats efficiently, increase revenue, and reduce expenses. Improving occupancy rates can be financially beneficial and achieved through pricing strategies and operational considerations. Airlines should focus on optimizing pricing strategies for gradual revenue growth.

##  Total revenue per year and the average revenue per ticket.

In [ ]:
revenue = pd.read_sql_query(""" 
                            SELECT aircraft_code, ticket_count, total_revenue,
                            total_revenue/ticket_count as AVG_REVENUE_PER_TICKET
                            FROM (
                            SELECT aircraft_code, COUNT(*) AS ticket_count,
                            sum(amount) AS TOTAL_REVENUE FROM ticket_flights
                            JOIN flights on ticket_flights.flight_id = flights.flight_id
                            GROUP BY aircraft_code)""", conn)
revenue.head()

In [ ]:
# Saving revenue data in csv file 

revenue.to_csv('revenue.csv')

### Calculate the average occupancy per aircraft

In [6]:
occupancy_rate = pd.read_sql_query("""select a.aircraft_code,avg(a.seats_count) as booked_seats, b.num_seats, avg(a.seats_count)/b.num_seats as occupancy_rate from
                (select aircraft_code,flights.flight_id,count(*) as seats_count from boarding_passes
                    inner join flights
                    on boarding_passes.flight_id = flights.flight_id
                    group by aircraft_code,flights.flight_id) as a
                    inner join 
                    (select aircraft_code,count(*) as num_seats from seats
                    group by aircraft_code) as b
                    on a.aircraft_code = b.aircraft_code group by a.aircraft_code""", conn
                  )
occupancy_rate

,aircraft_code,booked_seats,num_seats,occupancy_rate
0,319,53.583181,116,0.461924
1,321,88.809231,170,0.522407
2,733,80.255462,130,0.617350
3,763,113.937294,222,0.513231
4,773,264.925806,402,0.659019
5,CN1,6.004431,12,0.500369
6,CR2,21.482847,50,0.429657
7,SU9,56.812113,97,0.585692


In [ ]:
ocupancy_rate2 = pd.read_sql_query("""SELECT
    a.aircraft_code,
    AVG(a.seats_count) AS booked_seats,
    b.num_seats,
    AVG(a.seats_count) / b.num_seats AS occupancy_rate
FROM
    (SELECT
        aircraft_code,
        flights.flight_id,
        COUNT(*) AS seats_count
    FROM
        boarding_passes
    INNER JOIN
        flights ON boarding_passes.flight_id = flights.flight_id
    GROUP BY
        aircraft_code, flights.flight_id) AS a
INNER JOIN
    (SELECT
        aircraft_code,
        COUNT(*) AS num_seats
    FROM
        seats
    GROUP BY
        aircraft_code) AS b
ON
    a.aircraft_code = b.aircraft_code
GROUP BY
    a.aircraft_code;""", conn)

ocupancy_rate2


### Calculating how much the total annual turnover would increase by giving all aircraft a 10% higher occupancy rate.

In [ ]:
occupancy_rate['inc occupancy rate'] = occupancy_rate['occupancy_rate']+occupancy_rate['occupancy_rate']*0.1
occupancy_rate

pd.set_option("display.float_format",str)
total_revenue = pd.read_sql_query("""select aircraft_code,sum(amount) as total_revenue from ticket_flights
                        join flights on ticket_flights.flight_id = flights.flight_id
                        group by aircraft_code""", conn)
total_revenue

occupancy_rate['inc Total Annual Turnover'] = (total_revenue['total_revenue']/occupancy_rate['occupancy_rate'])*occupancy_rate['inc occupancy rate']
occupancy_rate